In [1]:
import re, html
import pandas as pd
from tqdm import tqdm

# ==== FILE INPUT / OUTPUT ====
INPUT_CSV = "youtube_comments_full.csv"
OUT_CSV   = "comments_clean_full.csv"

# ==== OPTIONS ====
USE_UNDERthesea = True      # Dùng tách từ tiếng Việt
DROP_STOPWORDS  = True      # Bỏ từ dừng (stopwords)
KEEP_NUMBERS    = True      # Giữ số (vd: "15pro", "13t")
KEEP_HASHTAG_MENTION = True # Giữ/tokens hashtag & mention
KEEP_URL_TOKEN  = True      # Giữ token cho URL (URL_TOKEN)

# ==== KIỂM TRA UNDERthesea ====
try:
    import underthesea
    HAS_UNDER = True
except Exception:
    HAS_UNDER = False

# ==== BIỂU THỨC REGEX ====
URL_RE       = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE     = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
HTML_TAGS_RE = re.compile(r"<[^>]+>")

# Giữ # và @ để còn bắt hashtag/mention
PUNC_KEEP_HASH_AT = r"[“”‘’\"'`.,;:!?(){}\[\]<>*~+=/\\|^$%]+"  # KHÔNG có # @
PUNC_RE   = re.compile(PUNC_KEEP_HASH_AT)

# Chỉ giữ chữ/số/khoảng trắng/underscore và ký tự Việt
KEEP_LATIN_VI = re.compile(r"[^0-9A-Za-zÀ-ỹ_\s#@]")

# Hashtag & Mention
HASHTAG_RE  = re.compile(r"(?<!\w)#([A-Za-z0-9_À-ỹ]+)")
MENTION_RE  = re.compile(r"(?<!\w)@([A-Za-z0-9_À-ỹ]+)")

# Nén ký tự lặp >2 (ví dụ: qúaaaa -> qúaá -> chuẩn hơn)
REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}", flags=re.UNICODE)

# ==== STOPWORDS (giữ lại “không”, “chưa”, “chả”) ====
VN_STOPWORDS = {
    "và","hoặc","là","của","cho","các","những","một","được","trong","khi","với",
    "đã","sẽ","tại","theo","này","kia","đó","thì","như","về","đến","từ","rằng",
    "đi","lên","xuống","nữa","nhiều","ít","rất","hơi","tôi","mình","bạn","chúng",
    "đấy","ấy","vậy","nhé","nhỉ"
}

# ==== TEENCODE / VIẾT TẮT CƠ BẢN ====
TEENCODE_MAP = {
    "ko":"không","k":"không","kh":"không","khong":"không",
    "hok":"không","hổng":"không","hem":"không",
    "đc":"được","dc":"được","đk":"được","dk":"được",
    "cx":"cũng","vs":"với","ms":"mới","mik":"mình",
    "ae":"anh em","crush":"crush","vl":"rất","vcl":"rất",
    "ad":"admin","ib":"inbox","rep":"trả lời",
}

# ==== EMOJI SENTIMENT MAP ====
EMOJI_MAP = {
    "👍":" positive ","❤":" positive ","❤️":" positive ","💖":" positive ","😍":" positive ",
    "😁":" positive ","😄":" positive ","😆":" positive ","👏":" positive ","🔥":" positive ",
    "🥰":" positive ",
    "👎":" negative ","😡":" negative ","🤬":" negative ","😠":" negative ","💔":" negative ",
    "😢":" negative ","😭":" negative ",
    "😂":" neutral ","😅":" neutral ","😐":" neutral ","😶":" neutral ","😊":" neutral "
}

def replace_emojis(text):
    for e, label in EMOJI_MAP.items():
        text = text.replace(e, label)
    return text

def normalize_teencode(tokens):
    return [TEENCODE_MAP.get(tok, tok) for tok in tokens]

def compress_repeats(s: str) -> str:
    # nén chuỗi "aaaa" -> "aa"
    return REPEAT_CHAR_RE.sub(r"\1\1", s)

def tokenize_hashtag_mention(s: str) -> str:
    # => HASHTAG_runningman, MENTION_user123
    s = HASHTAG_RE.sub(r" HASHTAG_\1 ", s)
    s = MENTION_RE.sub(r" MENTION_\1 ", s)
    return s

def clean_comment(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = html.unescape(s)
    s = replace_emojis(s)

    # URL → token hoặc xoá
    if KEEP_URL_TOKEN:
        s = URL_RE.sub(" URL_TOKEN ", s)
    else:
        s = URL_RE.sub(" ", s)

    s = EMAIL_RE.sub(" EMAIL_TOKEN ", s)
    s = HTML_TAGS_RE.sub(" ", s)

    s = s.lower()
    s = compress_repeats(s)

    # Giữ #/@ để token hóa trước khi bỏ dấu câu khác
    if KEEP_HASHTAG_MENTION:
        s = tokenize_hashtag_mention(s)

    # Bỏ dấu câu (trừ #/@ đã xử lý), giữ lại khoảng trắng
    s = PUNC_RE.sub(" ", s)

    # Bỏ ký tự ngoài bảng cho phép (giữ #/@ để token đã thay thế ở trên còn nghĩa)
    s = KEEP_LATIN_VI.sub(" ", s)

    # Bỏ số nếu cần
    if not KEEP_NUMBERS:
        s = re.sub(r"\d+", " ", s)

    # Chuẩn hoá khoảng trắng
    s = re.sub(r"\s{2,}", " ", s).strip()

    # Tách từ (tuỳ chọn)
    if USE_UNDERthesea and HAS_UNDER and s:
        s = underthesea.word_tokenize(s, format="text")

    # Teencode map (sau tách từ)
    if s:
        toks = s.split()
        toks = normalize_teencode(toks)
        # Stopwords (tránh xoá "không/chưa/chả")
        if DROP_STOPWORDS:
            toks = [w for w in toks if w not in VN_STOPWORDS]
        s = " ".join(toks)

    return s

# ==== XỬ LÝ FILE ====
print("📂 Đang đọc dữ liệu từ:", INPUT_CSV)
df = pd.read_csv(INPUT_CSV, dtype=str, low_memory=False)
if "text" not in df.columns:
    raise KeyError("Không tìm thấy cột 'text' trong file input.")

print("📊 Số dòng ban đầu:", len(df))

tqdm.pandas(desc="🧹 Cleaning comments")
df["text_clean"] = df["text"].fillna("").progress_apply(clean_comment)

# Loại comment trống sau khi clean
before = len(df)
df = df[df["text_clean"].str.strip() != ""]
after = len(df)
print(f"✅ Sau khi làm sạch: {after} comment hợp lệ (loại {before - after})")

# Lưu file mới (giữ nguyên các cột gốc)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("💾 Saved:", OUT_CSV, "| Columns:", list(df.columns))


📂 Đang đọc dữ liệu từ: youtube_comments_full.csv
📊 Số dòng ban đầu: 195755


🧹 Cleaning comments: 100%|██████████| 195755/195755 [00:03<00:00, 54505.00it/s]


✅ Sau khi làm sạch: 194980 comment hợp lệ (loại 775)
💾 Saved: comments_clean_full.csv | Columns: ['video_id', 'author', 'text', 'likeCount', 'publishedAt', 'text_clean']


# Gán nhãn

In [2]:
import pandas as pd

# ==== FILE PATH ====
COMMENTS_FILE = "comments_clean_full.csv"
VIDEOS_FILE   = "crawl_videos_merge.csv"
OUT_FILE      = "comments_clean_full_with_brand.csv"

# ==== ĐỌC DỮ LIỆU ====
print("📂 Đang đọc dữ liệu...")
comments = pd.read_csv(COMMENTS_FILE, dtype=str, low_memory=False)
videos   = pd.read_csv(VIDEOS_FILE, dtype=str, low_memory=False)

# ==== KIỂM TRA CỘT ====
required_cols = {"video_id", "brand"}
missing = required_cols - set(videos.columns)
if missing:
    raise KeyError(f"❌ Thiếu cột {missing} trong file {VIDEOS_FILE}")

if "video_id" not in comments.columns:
    raise KeyError(f"❌ Thiếu cột 'video_id' trong file {COMMENTS_FILE}")

print("📊 Số comment:", len(comments))
print("📊 Số video có brand:", len(videos))

# ==== GÁN BRAND THEO VIDEO_ID ====
comments = comments.merge(
    videos[["video_id", "brand"]],
    on="video_id",
    how="left"
)

# ==== THỐNG KÊ ====
matched = comments["brand"].notna().sum()
print(f"✅ Đã gán nhãn brand cho {matched:,}/{len(comments):,} comment ({matched/len(comments)*100:.1f}%)")

# ==== KIỂM TRA CÁC GIÁ TRỊ BRAND ====
print("🏷️ Các brand hiện có:", comments["brand"].dropna().unique())

# ==== LƯU FILE ====
comments.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
print("💾 Đã lưu file:", OUT_FILE)

📂 Đang đọc dữ liệu...
📊 Số comment: 194980
📊 Số video có brand: 8167
✅ Đã gán nhãn brand cho 194,980/194,980 comment (100.0%)
🏷️ Các brand hiện có: ['Apple' 'Samsung' 'OPPO' 'Xiaomi']
💾 Đã lưu file: comments_clean_full_with_brand.csv


# Data audit sau clean

In [3]:
# =========================================================
# DATA AUDIT (sau clean + gán brand)
# - Giữ comment ngắn nhưng có ý nghĩa (mê/đẹp/ok, emoji, 15pro...)
# - Khử trùng gần (MinHash nếu có, fallback simple)
# - Tính engagement_score
# - Chuẩn hóa thời gian: UTC + Asia/Ho_Chi_Minh (fallback nếu thiếu tzdata)
# - Lưu cả Parquet & CSV
# =========================================================
import re
import json
import pandas as pd
import numpy as np

# -------- CONFIG --------
IN_FILE   = "comments_clean_full_with_brand.csv"   

SAVE_PARQUET = True
SAVE_CSV     = True
PARQUET_OUT  = "comments_ready.parquet"
CSV_OUT      = "comments_ready.csv"

# -------- READ & RENAME --------
df = pd.read_csv(IN_FILE, dtype=str, low_memory=False)
n0 = len(df)

# 🧩 Chuẩn hoá tên cột cho đúng (dựa theo file bạn cung cấp)
rename_map = {
    "likeCount": "like_count",
    "replyCount": "reply_count",   # nếu không có thì tạo sau
    "publishedAt": "published_at"
}
df.rename(columns=rename_map, inplace=True)

# Nếu file không có reply_count, thêm cột 0 mặc định
if "reply_count" not in df.columns:
    df["reply_count"] = 0

# Chuẩn hóa cột bắt buộc
required = {"video_id", "brand", "text_clean"}
missing_req = list(required - set(df.columns))
if missing_req:
    raise KeyError(f"Thiếu cột bắt buộc: {missing_req}. Hãy chắc file đã clean và gán brand.")

# Ép kiểu số cho like/reply
for c in ["like_count", "reply_count"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    else:
        df[c] = 0

# Điền thiếu cơ bản
df["text_clean"] = df["text_clean"].fillna("").astype(str)
df["brand"] = df["brand"].fillna("unknown")

# Lọc thiếu video_id
df = df.dropna(subset=["video_id"]).copy()
n_after_drop_vid = len(df)

# -------- MEANINGFUL FILTER --------
SHORT_MEANINGFUL = {
    "mê","đẹp","xấu","tệ","đỉnh","đã","xịn","phê","gắt","chất","ổn",
    "hay","chán","dở","bựa","kì","đơ","lag","lỗi","ok","oke","oki",
    "wow","ê","ơ","cute","xỉu"
}
EMOJI_TOKENS = {"positive","negative","neutral"}  # do bước clean đã map emoji -> 3 token này
ALNUM_MIX_RE = re.compile(r"(?=.*[A-Za-z])(?=.*\d)")  # như 15pro, s24u, 14pm

def is_meaningful(row):
    txt = (row.get("text_clean") or "").strip()
    if not txt:
        return False
    toks = txt.split()
    # 1) ≥ 2 token
    if len(toks) >= 2:
        return True
    # 2) 1 token nhưng có ý nghĩa
    if len(toks) == 1:
        t = toks[0]
        if t in SHORT_MEANINGFUL or t in EMOJI_TOKENS or ALNUM_MIX_RE.search(t):
            return True
    # 3) Có like >= 1 thì giữ
    try:
        like = int(row.get("like_count", 0))
    except Exception:
        like = 0
    return like >= 1

df["_keep_meaning"] = df.apply(is_meaningful, axis=1)
n_kept_meaning = int(df["_keep_meaning"].sum())
df = df[df["_keep_meaning"]].drop(columns=["_keep_meaning"]).copy()

# -------- NEAR-DUPLICATE REMOVAL --------
def simple_signature(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

USE_MINHASH = True
dedup_method = "simple"

if USE_MINHASH:
    try:
        from datasketch import MinHash, MinHashLSH
        dedup_method = "minhash"
    except Exception:
        dedup_method = "simple"

if dedup_method == "minhash":
    df = df.reset_index(drop=True)
    def mhash_tokens(text, num_perm=64):
        m = MinHash(num_perm=num_perm)
        for tok in text.split():
            m.update(tok.encode("utf8"))
        return m

    num_perm = 64
    threshold = 0.90
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    mh_list = []
    for i, txt in enumerate(df["text_clean"]):
        mh = mhash_tokens(txt or "", num_perm=num_perm)
        lsh.insert(f"idx_{i}", mh)
        mh_list.append(mh)

    eng = df["like_count"] + 0.5 * df["reply_count"]
    order = np.argsort(-eng.values)
    keep_mask = np.ones(len(df), dtype=bool)
    visited = set()

    for i in order:
        if not keep_mask[i]:
            continue
        key_i = f"idx_{i}"
        if key_i in visited:
            continue
        neigh = lsh.query(mh_list[i])
        for nkey in neigh:
            if nkey == key_i:
                continue
            j = int(nkey.split("_", 1)[1])
            keep_mask[j] = False
        visited.add(key_i)

    n_before_dup = len(df)
    df = df[keep_mask].copy().reset_index(drop=True)
    n_after_dup = len(df)
else:
    df["__sig"] = df["text_clean"].map(simple_signature)
    n_before_dup = len(df)
    df = df.sort_values(["like_count", "reply_count"], ascending=[False, False]) \
           .drop_duplicates(subset="__sig", keep="first") \
           .drop(columns="__sig") \
           .reset_index(drop=True)
    n_after_dup = len(df)

# -------- ENGAGEMENT SCORE --------
ENGAGEMENT_FORMULA = "linear"
if ENGAGEMENT_FORMULA == "linear":
    df["engagement_score"] = df["like_count"] + 0.5 * df["reply_count"]
else:
    df["engagement_score"] = np.log1p(df["like_count"]) + 0.7 * np.log1p(df["reply_count"])

# -------- TIME NORMALIZATION --------
if "published_at" in df.columns:
    ts_utc = pd.to_datetime(df["published_at"], errors="coerce", utc=True)
    df["published_dt_utc"] = ts_utc
    try:
        ts_vn = ts_utc.dt.tz_convert("Asia/Ho_Chi_Minh")
        df["date_vn"]  = ts_vn.dt.date.astype(str)
        df["week_vn"]  = ts_vn.dt.isocalendar().week.astype("Int64")
        df["year_vn"]  = ts_vn.dt.year.astype("Int64")
        df["month_vn"] = ts_vn.dt.to_period("M").astype(str)
    except Exception:
        ts_naive = ts_utc.dt.tz_localize(None)
        ts_vn2 = ts_naive + pd.Timedelta(hours=7)
        df["date_vn"]  = ts_vn2.dt.date.astype(str)
        df["week_vn"]  = ts_vn2.dt.isocalendar().week.astype("Int64")
        df["year_vn"]  = ts_vn2.dt.year.astype("Int64")
        df["month_vn"] = ts_vn2.dt.to_period("M").astype(str)
else:
    for c in ["published_dt_utc","date_vn","week_vn","year_vn","month_vn"]:
        df[c] = pd.NA

# -------- SAVE --------
if SAVE_PARQUET:
    df.to_parquet(PARQUET_OUT, index=False)
if SAVE_CSV:
    import csv
    df.to_csv(
        CSV_OUT,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_MINIMAL,
        lineterminator="\n"
    )

# -------- REPORT --------
report = {
    "input_rows": n0,
    "after_drop_missing_video_id": int(n_after_drop_vid),
    "kept_by_semantic_filter": int(n_kept_meaning),
    "after_dedup": int(n_after_dup),
    "dedup_method": dedup_method,
    "brand_distribution": df["brand"].value_counts(dropna=False).to_dict(),
    "saved": {
        "parquet": PARQUET_OUT if SAVE_PARQUET else None,
        "csv": CSV_OUT if SAVE_CSV else None
    }
}
print("---- DATA AUDIT SUMMARY ----")
print(json.dumps(report, ensure_ascii=False, indent=2))


C:\Users\HOME\AppData\Local\Temp\ipykernel_9400\1602546003.py:172: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month_vn"] = ts_vn.dt.to_period("M").astype(str)


---- DATA AUDIT SUMMARY ----
{
  "input_rows": 194980,
  "after_drop_missing_video_id": 194980,
  "kept_by_semantic_filter": 192960,
  "after_dedup": 186744,
  "dedup_method": "simple",
  "brand_distribution": {
    "Apple": 58042,
    "Xiaomi": 55253,
    "Samsung": 51308,
    "OPPO": 22141
  },
  "saved": {
    "parquet": "comments_ready.parquet",
    "csv": "comments_ready.csv"
  }
}
